In [ ]:
import os, sys, joblib
import numpy as np

sys.path.append("..")

from evaluation.utils.evaluation_functions import load_and_prepare_data_xgb, load_and_prepare_data_baseline
from evaluation.utils.calculate_tables import (                            
    build_all_tables_for_model
)

CALIBRATED_MODELS_DIR = "../models/calibrated"
FIGURES_DIR = "figures"

In [ ]:
class_name = "baseline" # <-- select model

### Help Functions

In [ ]:
# Load validation and test splits from the DB (your function)
x_val, y_val, x_test, y_test, feat_cols = None, None, None, None, None
if class_name == "baseline":
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_baseline()
else:
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_xgb(
        table_name=f"merged_{class_name}_features",
        drop_treatment_given=True,
        drop_only_2_values=True
    )
    
y_val  = np.asarray(y_val).ravel()
y_test = np.asarray(y_test).ravel()

### Load model

In [ ]:
# Load calibrated model
if class_name == "baseline":
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr.pkl"))
else:
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, f"xgb_{class_name}.pkl"))

# Use positive-class probabilities
y_val_pred  = model.predict_proba(x_val)
y_test_pred = model.predict_proba(x_test)
y_val_scores  = y_val_pred[:, 1] if y_val_pred.ndim == 2 else y_val_pred
y_test_scores = y_test_pred[:, 1] if y_test_pred.ndim == 2 else y_test_pred

### Calculate Metric Tables

In [ ]:
tables = build_all_tables_for_model(
    class_name=class_name,
    y_val=y_val,          y_val_pred=y_val_scores,
    y_test=y_test,        y_test_pred=y_test_scores,
    targets=(0.70, 0.75, 0.80, 0.85, 0.90),
    main_target=0.80, 
    n_boot=1000, alpha=0.95, seed=42,
    figures_dir=FIGURES_DIR,
    strategy="quantile_pos",
    rule=">=",
)

In [ ]:
tables